# Демо: логирование DistilBERT в MLflow

Модель: `distilbert-base-uncased` (легкая и стабильная для обучения).
Задача: бинарная классификация текстов.

Пайплайн:
1. Подготовка мини-датасета
2. Fine-tuning DistilBERT (1 эпоха)
3. Логирование метрик и модели в MLflow
4. Пометка версии как `PRD` и загрузка по alias `prd`

In [4]:
import os
import random
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["MLFLOW_TRACKING_URI"] = os.getenv("MLFLOW_TRACKING_URI", "http://70.34.242.179")
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

device = torch.device("cuda" if torch.cuda.is_available() else "mps")
print("Device:", device)

Device: mps


## 1) Мини-датасет

In [6]:
positive = [
    "I love this product", "Great quality and fast delivery", "Amazing experience",
    "Very happy with the purchase", "Works perfectly", "Excellent support team",
    "Totally worth the money", "Super useful and simple", "Best thing I bought this month",
    "I would recommend it to everyone"
]
negative = [
    "I hate this product", "Terrible quality", "Very disappointed",
    "Waste of money", "It stopped working in one day", "Support was not helpful",
    "Awful experience", "Not as described", "I want a refund",
    "Worst purchase ever"
]

texts = positive + negative
labels = [1] * len(positive) + [0] * len(negative)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=SEED, stratify=labels
)

print("train size:", len(X_train), "test size:", len(X_test))

train size: 14 test size: 6


In [7]:
MODEL_NAME = "distilbert-base-uncased"

# use_fast=False снижает риск проблем с backend-tokenizer в учебной среде
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
class TextClsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

## 2) Обучение + MLflow

In [9]:
experiment_name = "students-bert-tiny-demo-proxy"
artifact_location = "mlflow-artifacts:/"
registered_model_name = "students_bert_tiny_sentiment"

client = MlflowClient()
exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(name=experiment_name, artifact_location=artifact_location)
else:
    exp_id = exp.experiment_id

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1779266125774, experiment_id='1', last_update_time=1779266125774, lifecycle_stage='active', name='students-bert-tiny-demo-proxy', tags={}, trace_location=None, workspace='default'>

In [13]:
train_ds = TextClsDataset(X_train, y_train, tokenizer)
test_ds = TextClsDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

params = {
    "model_name": MODEL_NAME,
    "epochs": 1,
    "lr": 1e-4,
    "batch_size": 8,
    "max_len": 64
}

optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"])
num_training_steps = params["epochs"] * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
with mlflow.start_run(experiment_id=exp_id, run_name="bert_tiny_baseline"):
    mlflow.log_params(params)

    model.train()
    running_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    train_loss = running_loss / max(len(train_loader), 1)
    mlflow.log_metric("train_loss", float(train_loss))

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    metrics = {
        "test_accuracy": float(accuracy_score(all_labels, all_preds)),
        "test_f1": float(f1_score(all_labels, all_preds)),
    }
    mlflow.log_metrics(metrics)

    task_components = {
        "model": model,
        "tokenizer": tokenizer,
    }
    model_info = mlflow.transformers.log_model(
        transformers_model=task_components,
        name="model",
        task="text-classification",
        registered_model_name=registered_model_name,
    )

    new_version = model_info.registered_model_version
    client.set_model_version_tag(registered_model_name, new_version, "env", "PRD")
    client.set_registered_model_alias(registered_model_name, "prd", new_version)

    print("Run ID:", mlflow.active_run().info.run_id)
    print("Registered model version:", new_version)
    print("Alias 'prd' points to version:", new_version)
    print("Metrics:", metrics)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026/05/20 11:21:28 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /var/folders/7k/l7vx_5fj6c178hfh4sgw4jvc0000gn/T/tmpykl9xg_2/model, flavor: transformers). Fall back to return ['transformers==5.3.0', 'torch==2.9.1', 'torchvision==0.25.0']. Set logging level to DEBUG to see the full traceback. 
Registered model 'students_bert_tiny_sentiment' already exists. Creating a new version of this model...
2026/05/20 11:21:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: students_bert_tiny_sentiment, version 2
Created version '2' of model 'students_bert_tiny_sentiment'.


Run ID: 74eb27b11f3a4729a5d622a2a0c6762f
Registered model version: 2
Alias 'prd' points to version: 2
Metrics: {'test_accuracy': 0.5, 'test_f1': 0.0}
🏃 View run bert_tiny_baseline at: http://70.34.242.179/#/experiments/1/runs/74eb27b11f3a4729a5d622a2a0c6762f
🧪 View experiment at: http://70.34.242.179/#/experiments/1


## 3) Загрузка модели по alias `prd`

In [15]:
loaded = mlflow.transformers.load_model(f"models:/{registered_model_name}@prd")

sample_texts = [
    "excellent product and quick delivery",
    "very bad quality and terrible service"
]

preds = loaded(sample_texts)
preds

2026/05/20 11:22:58 INFO mlflow.transformers: 'models:/students_bert_tiny_sentiment@prd' resolved as 'mlflow-artifacts:/models/m-6f29d1b947544bd58cf2849b8edeed41/artifacts'


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


[{'label': 'LABEL_1', 'score': 0.5282853245735168},
 {'label': 'LABEL_0', 'score': 0.5605860352516174}]